In [4]:
# ============================================================
# WHAM POSE ESTIMATION FOR SUBJECT EXPERIMENT VIDEOS
# One-cell Kaggle version
# ============================================================
#
# Input example:
# /kaggle/input/<your-dataset>/SUB01.zip
# or:
# /kaggle/input/<your-dataset>/SUB01/E0/SUB01_E0_wham_rgb_150fps.mp4
# /kaggle/input/<your-dataset>/SUB01/E1/SUB01_E1_wham_rgb_150fps.mp4
# ...
#
# Output:
# /kaggle/working/wham_pkls/SUB01/E0/SUB01_E0_wham_output.pkl
# /kaggle/working/wham_pkls/SUB01/E1/SUB01_E1_wham_output.pkl
# ...
# /kaggle/working/SUB01_wham_pkls.zip
#
# ============================================================

from pathlib import Path
import os
import subprocess
import shutil
import zipfile
import re
import sys
import time
import pandas as pd


# ============================================================
# USER SETTINGS
# ============================================================

KAGGLE_INPUT_ROOT = Path("/kaggle/input/datasets/sophie160/sub17-experiments")

def infer_subject_from_input_root(input_root: Path) -> str:
    """
    Infer subject automatically from folder or zip names.
    Looks for names like SUB01, SUB02, SUB08, SUB17.
    """
    subject_pattern = re.compile(r"SUB\d+", re.IGNORECASE)

    candidates = []

    for p in input_root.rglob("*"):
        match = subject_pattern.search(p.name)
        if match:
            candidates.append(match.group(0).upper())

    candidates = sorted(set(candidates))

    if not candidates:
        raise ValueError(
            f"Could not infer subject from input folder:\n{input_root}\n"
            "Expected a folder or zip name containing something like SUB01."
        )

    if len(candidates) > 1:
        raise ValueError(
            f"Multiple subjects found under input folder: {candidates}\n"
            "Use one subject per Kaggle run, or explicitly set SUBJECT."
        )

    return candidates[0]


SUBJECT = infer_subject_from_input_root(KAGGLE_INPUT_ROOT)

# WHAM settings
USE_ESTIMATE_LOCAL_ONLY = True

# Working folders
WORKING_DIR = Path("/kaggle/working")
WHAM_DIR = WORKING_DIR / "WHAM"
CONDA_DIR = WORKING_DIR / "miniconda"
VIDEO_WORK_ROOT = WORKING_DIR / "videos"
PKL_OUTPUT_ROOT = WORKING_DIR / "wham_pkls"

# Conda executable paths
CONDA = CONDA_DIR / "bin" / "conda"
PYTHON = CONDA_DIR / "envs" / "wham" / "bin" / "python"

# Output folder for subject-specific PKLs
SUBJECT_PKL_DIR = PKL_OUTPUT_ROOT / SUBJECT

VIDEO_WORK_ROOT.mkdir(parents=True, exist_ok=True)
SUBJECT_PKL_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def run(cmd, cwd=None, env=None, check=True):
    """
    Runs a shell/list command and raises an error if it fails.
    """
    print("\n" + "=" * 90)
    print("Running command:")
    if isinstance(cmd, list):
        print(" ".join(str(c) for c in cmd))
    else:
        print(cmd)
    print("=" * 90)

    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        shell=isinstance(cmd, str),
        text=True,
    )

    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with return code {result.returncode}")

    return result


def experiment_number_from_path(path: Path) -> int:
    """
    Extracts experiment number from folder or filename.

    Works for:
    /.../SUB01/E0/SUB01_E0_wham_rgb_150fps.mp4
    /.../SUB01/E13/SUB01_E13_wham_rgb_150fps.mp4
    /.../SUB01_E3_wham_output.pkl
    """
    path = Path(path)

    for part in path.parts:
        match = re.fullmatch(r"E(\d+)", part)
        if match:
            return int(match.group(1))

    match = re.search(r"_E(\d+)_", path.name)
    if match:
        return int(match.group(1))

    match = re.search(r"E(\d+)", path.name)
    if match:
        return int(match.group(1))

    raise ValueError(f"Could not infer experiment number from path: {path}")


def find_subject_video_root(subject: str) -> Path:
    """
    Finds the subject video folder.

    Handles two cases:
    1. Kaggle has extracted folder:
       /kaggle/input/.../SUB01/E0/*.mp4

    2. Kaggle has zip:
       /kaggle/input/.../SUB01.zip
       This gets extracted to /kaggle/working/videos/SUB01
    """

    print("\nSearching for subject videos...")

    # Case 1: already extracted subject folder
    extracted_candidates = [
        p for p in KAGGLE_INPUT_ROOT.rglob(subject)
        if p.is_dir()
    ]

    if extracted_candidates:
        selected = extracted_candidates[0]
        print("Found extracted subject folder:")
        print(selected)
        return selected

    # Case 2: subject zip
    zip_candidates = [
        p for p in KAGGLE_INPUT_ROOT.rglob(f"{subject}.zip")
        if p.is_file()
    ]

    if not zip_candidates:
        raise FileNotFoundError(
            f"Could not find either an extracted {subject}/ folder or {subject}.zip under:\n"
            f"{KAGGLE_INPUT_ROOT}"
        )

    zip_path = zip_candidates[0]

    print("Found zip file:")
    print(zip_path)

    # Extract into working directory
    extract_base = VIDEO_WORK_ROOT

    print("Extracting zip to:")
    print(extract_base)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_base)

    # Expected structure after extraction
    expected_subject_dir = extract_base / subject

    if expected_subject_dir.exists():
        print("Extracted subject folder:")
        print(expected_subject_dir)
        return expected_subject_dir

    # Fallback: search extracted folder
    extracted_after = [
        p for p in extract_base.rglob(subject)
        if p.is_dir()
    ]

    if extracted_after:
        selected = extracted_after[0]
        print("Found extracted subject folder after fallback search:")
        print(selected)
        return selected

    raise FileNotFoundError(
        f"Zip was extracted, but no {subject}/ folder was found inside it.\n"
        f"Check the internal structure of: {zip_path}"
    )


def find_subject_videos(subject_video_root: Path):
    """
    Finds all mp4/mov videos under the subject folder and sorts by E number.
    """
    videos = (
        list(subject_video_root.rglob("*.mp4")) +
        list(subject_video_root.rglob("*.MP4")) +
        list(subject_video_root.rglob("*.mov")) +
        list(subject_video_root.rglob("*.MOV"))
    )

    videos = sorted(videos, key=experiment_number_from_path)

    if not videos:
        raise FileNotFoundError(f"No videos found under:\n{subject_video_root}")

    print("\nVideos found in experiment order:")
    for v in videos:
        exp_num = experiment_number_from_path(v)
        print(f"E{exp_num}: {v}")

    return videos


def get_expected_wham_pkl(video_path: Path) -> Path:
    """
    WHAM usually saves:
    /kaggle/working/WHAM/output/demo/<video_stem>/wham_output.pkl
    """
    return WHAM_DIR / "output" / "demo" / video_path.stem / "wham_output.pkl"


def newest_wham_pkl() -> Path:
    """
    Fallback: finds newest wham_output.pkl in WHAM/output.
    """
    candidates = sorted(
        (WHAM_DIR / "output").rglob("wham_output.pkl"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    if not candidates:
        raise FileNotFoundError("No wham_output.pkl files found in WHAM/output.")

    return candidates[0]


# ============================================================
# 1. INSTALL MINICONDA + CREATE WHAM ENVIRONMENT
# ============================================================

os.chdir(WORKING_DIR)

if not CONDA.exists():
    print("\nInstalling Miniconda...")
    run("wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh")
    run("bash miniconda.sh -b -p /kaggle/working/miniconda")
else:
    print("\nMiniconda already exists.")

# Accept conda terms if needed
run([str(CONDA), "tos", "accept", "--override-channels", "--channel", "https://repo.anaconda.com/pkgs/main"], check=False)
run([str(CONDA), "tos", "accept", "--override-channels", "--channel", "https://repo.anaconda.com/pkgs/r"], check=False)

if not PYTHON.exists():
    print("\nCreating WHAM conda environment...")
    run([str(CONDA), "create", "-y", "-n", "wham", "python=3.9", "pip"])
else:
    print("\nWHAM conda environment already exists.")

run([str(PYTHON), "--version"])


# ============================================================
# 2. CLONE WHAM
# ============================================================

if not WHAM_DIR.exists():
    print("\nCloning WHAM...")
    run(["git", "clone", "https://github.com/yohanshin/WHAM.git", "--recursive"], cwd=WORKING_DIR)
else:
    print("\nWHAM directory already exists.")

run(["git", "submodule", "update", "--init", "--recursive"], cwd=WHAM_DIR)


# ============================================================
# 3. INSTALL WHAM DEPENDENCIES
# ============================================================

print("\nInstalling Python dependencies...")

run([str(PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

# PyTorch for Kaggle GPU
run([
    str(PYTHON), "-m", "pip", "install",
    "torch==1.10.2+cu113",
    "torchvision==0.11.3+cu113",
    "torchaudio==0.10.2+cu113",
    "--extra-index-url", "https://download.pytorch.org/whl/cu113"
])

# General dependencies
run([
    str(PYTHON), "-m", "pip", "install",
    "numpy==1.22.3",
    "Cython<3",
    "joblib",
    "scipy",
    "six",
    "yacs",
    "scikit-image",
    "opencv-python",
    "matplotlib",
    "tensorboard",
    "smplx",
    "progress",
    "einops",
    "loguru",
    "requests",
    "psutil",
    "tqdm",
    "timm",
    "pandas",
    "pyyaml",
    "seaborn",
    "thop",
])

# chumpy compatibility
run([
    str(PYTHON), "-m", "pip", "install",
    "chumpy @ git+https://github.com/mattloper/chumpy",
    "--no-build-isolation"
])

# mmcv
run([
    str(PYTHON), "-m", "pip", "install",
    "mmcv-full==1.5.0",
    "-f", "https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html"
])

# Ultralytics without pulling newer dependencies
run([
    str(PYTHON), "-m", "pip", "install",
    "ultralytics==8.0.145",
    "--no-deps"
])

# Stabilize setuptools
run([
    str(PYTHON), "-m", "pip", "install",
    "--upgrade",
    "setuptools<70"
])


# ============================================================
# 4. INSTALL VITPOSE
# ============================================================

VITPOSE_DIR = WHAM_DIR / "third-party" / "ViTPose"

if not VITPOSE_DIR.exists():
    raise FileNotFoundError(f"ViTPose directory not found:\n{VITPOSE_DIR}")

run([str(PYTHON), "-m", "pip", "install", "-e", "."], cwd=VITPOSE_DIR)


# ============================================================
# 5. CHECK CUDA / TORCH / MMCV
# ============================================================

print("\nChecking GPU and key packages...")

run([
    str(PYTHON), "-c",
    "import torch; "
    "print('torch:', torch.__version__); "
    "print('cuda available:', torch.cuda.is_available()); "
    "print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')"
])

run([
    str(PYTHON), "-c",
    "import mmcv; print('mmcv:', mmcv.__version__)"
])


# ============================================================
# 6. FETCH WHAM MODEL DATA
# ============================================================
#
# This requires Kaggle Secrets:
# SMPLIFY_USERNAME
# SMPLIFY_PASSWORD
# SMPL_USERNAME
# SMPL_PASSWORD
#
# In Kaggle:
# Add-ons > Secrets > Add these four secrets
#
# If you already have the WHAM data/models present, this may be skipped.
# ============================================================

print("\nTrying to fetch WHAM demo/model data...")

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    answers = [
        secrets.get_secret("SMPLIFY_USERNAME"),
        secrets.get_secret("SMPLIFY_PASSWORD"),
        secrets.get_secret("SMPL_USERNAME"),
        secrets.get_secret("SMPL_PASSWORD"),
        secrets.get_secret("SMPLIFY_USERNAME"),
        secrets.get_secret("SMPLIFY_PASSWORD"),
        secrets.get_secret("SMPL_USERNAME"),
        secrets.get_secret("SMPL_PASSWORD"),
    ]

    input_text = "\n".join(answers) + "\n"

    subprocess.run(
        ["bash", "fetch_demo_data.sh"],
        cwd=str(WHAM_DIR),
        input=input_text,
        text=True,
        check=True,
    )

    print("WHAM demo/model data fetched successfully.")

except Exception as e:
    print("\nWARNING: Could not fetch WHAM demo/model data automatically.")
    print("Reason:", repr(e))
    print("\nIf WHAM later fails because models are missing, check your Kaggle Secrets:")
    print("- SMPLIFY_USERNAME")
    print("- SMPLIFY_PASSWORD")
    print("- SMPL_USERNAME")
    print("- SMPL_PASSWORD")


# ============================================================
# 7. PATCH COMPATIBILITY ISSUES
# ============================================================

print("\nPatching compatibility issues...")

# Patch .mT for older Torch compatibility
patched_files = []

for file_path in WHAM_DIR.rglob("*.py"):
    try:
        text = file_path.read_text(errors="ignore")
    except Exception:
        continue

    if ".mT" in text:
        new_text = text.replace(".mT", ".transpose(-2, -1)")
        file_path.write_text(new_text)
        patched_files.append(file_path)

if patched_files:
    print("Patched .mT in these files:")
    for f in patched_files:
        print(f)
else:
    print("No .mT patches needed.")

# Patch tensorboard distutils issue
torch_tb_init = (
    CONDA_DIR
    / "envs"
    / "wham"
    / "lib"
    / "python3.9"
    / "site-packages"
    / "torch"
    / "utils"
    / "tensorboard"
    / "__init__.py"
)

if torch_tb_init.exists():
    text = torch_tb_init.read_text(errors="ignore")
    if "import distutils.version" not in text:
        text = text.replace("import distutils", "import distutils\nimport distutils.version")
        torch_tb_init.write_text(text)
        print("Patched tensorboard distutils issue.")
    else:
        print("Tensorboard distutils patch already present.")
else:
    print("Tensorboard init file not found; skipping tensorboard patch.")

run([
    str(PYTHON), "-c",
    "from torch.utils.tensorboard import SummaryWriter; print('tensorboard import works')"
])


# ============================================================
# 8. FIND SUBJECT VIDEOS
# ============================================================

subject_video_root = find_subject_video_root(SUBJECT)
videos = find_subject_videos(subject_video_root)


# ============================================================
# 9. RUN WHAM FOR EACH EXPERIMENT AND SAVE UNIQUE PKL
# ============================================================

print("\nStarting WHAM processing...")

env = os.environ.copy()
env["PYTHONPATH"] = str(VITPOSE_DIR) + ":" + env.get("PYTHONPATH", "")

summary_rows = []

for video_path in videos:
    video_path = Path(video_path)
    exp_num = experiment_number_from_path(video_path)
    exp_name = f"E{exp_num}"

    print("\n" + "#" * 90)
    print(f"Processing {SUBJECT} {exp_name}")
    print(f"Video: {video_path}")
    print("#" * 90)

    before_time = time.time()

    cmd = [
        str(PYTHON),
        "demo.py",
        "--video",
        str(video_path),
        "--save_pkl",
    ]

    if USE_ESTIMATE_LOCAL_ONLY:
        cmd.append("--estimate_local_only")

    run(cmd, cwd=WHAM_DIR, env=env)

    # Expected WHAM pkl
    expected_pkl = get_expected_wham_pkl(video_path)

    if expected_pkl.exists():
        source_pkl = expected_pkl
        print("\nFound expected WHAM PKL:")
        print(source_pkl)
    else:
        print("\nWARNING: Expected WHAM PKL was not found:")
        print(expected_pkl)
        print("Using newest wham_output.pkl as fallback.")

        source_pkl = newest_wham_pkl()

        if source_pkl.stat().st_mtime < before_time:
            raise RuntimeError(
                f"Newest WHAM PKL appears older than the current run:\n"
                f"{source_pkl}\n"
                "This may mean WHAM did not generate a new PKL."
            )

        print("Fallback PKL:")
        print(source_pkl)

    # Output folder for this experiment
    exp_pkl_dir = SUBJECT_PKL_DIR / exp_name
    exp_pkl_dir.mkdir(parents=True, exist_ok=True)

    # Final stable metrics PKL
    final_pkl = exp_pkl_dir / f"{SUBJECT}_{exp_name}_wham_output.pkl"

    shutil.copy2(source_pkl, final_pkl)

    print("\nSaved experiment-specific PKL:")
    print(final_pkl)

    summary_rows.append({
        "subject": SUBJECT,
        "experiment": exp_name,
        "experiment_number": exp_num,
        "video_path": str(video_path),
        "wham_original_pkl": str(source_pkl),
        "metrics_pkl": str(final_pkl),
    })


# ============================================================
# 10. SAVE SUMMARY CSV
# ============================================================

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("experiment_number")

summary_csv = SUBJECT_PKL_DIR / f"{SUBJECT}_wham_pkl_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("\nSaved summary CSV:")
print(summary_csv)

print("\nSummary:")
display(summary_df)


# ============================================================
# 11. ZIP OUTPUT PKLS FOR DOWNLOAD
# ============================================================

zip_base = WORKING_DIR / f"{SUBJECT}_wham_pkls"
zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=SUBJECT_PKL_DIR
)

print("\nCreated downloadable ZIP:")
print(zip_path)


# ============================================================
# 12. FINAL OUTPUT CHECK
# ============================================================

print("\nFinal PKL files for metrics:")

final_pkls = sorted(
    SUBJECT_PKL_DIR.rglob("*.pkl"),
    key=experiment_number_from_path
)

for p in final_pkls:
    print(p)

print("\nDone.")
print(f"Use this ZIP for download: {zip_path}")


Miniconda already exists.

Running command:
/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/main

Running command:
/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
accepted Terms of Service for https://repo.anaconda.com/pkgs/r

WHAM conda environment already exists.

Running command:
/kaggle/working/miniconda/envs/wham/bin/python --version
Python 3.9.25

WHAM directory already exists.

Running command:
git submodule update --init --recursive

Installing Python dependencies...

Running command:
/kaggle/working/miniconda/envs/wham/bin/python -m pip install --upgrade pip setuptools wheel
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 69.5.1
    Unins

  Running command git clone --filter=blob:none --quiet https://github.com/mattloper/chumpy /tmp/pip-install-m4dasekk/chumpy_5a8359a0bc2343ed907db8cba18f2bad


  Resolved https://github.com/mattloper/chumpy to commit 580566eafc9ac68b2614b64d6f7aaa84eebb70da
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'

Running command:
/kaggle/working/miniconda/envs/wham/bin/python -m pip install mmcv-full==1.5.0 -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html
Looking in links: https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html

Running command:
/kaggle/working/miniconda/envs/wham/bin/python -m pip install ultralytics==8.0.145 --no-deps

Running command:
/kaggle/working/miniconda/envs/wham/bin/python -m pip install --upgrade setuptools<70
  Using cached setuptools-69.5.1-py3-none-any.whl.metadata (6.2 kB)
Using cached setuptools-69.5.1-py3-none-any.whl (894 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1

Running command:
/

--2026-06-23 16:36:35--  https://download.is.tue.mpg.de/download.php?domain=smplify&resume=1&sfile=mpips_smplify_public_v2.zip
Resolving download.is.tue.mpg.de (download.is.tue.mpg.de)... 192.124.27.139
Connecting to download.is.tue.mpg.de (download.is.tue.mpg.de)|192.124.27.139|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: download.php?domain=smplify&sfile=mpips_smplify_public_v2.zip&resume=1 [following]
--2026-06-23 16:36:36--  https://download.is.tue.mpg.de/download.php?domain=smplify&sfile=mpips_smplify_public_v2.zip&resume=1
Reusing existing connection to download.is.tue.mpg.de:443.
HTTP request sent, awaiting response... 200 OK
Length: 53028966 (51M) [application/octet-stream]
Saving to: ‘./dataset/body_models/smplify.zip’

     0K .......... .......... .......... .......... ..........  0%  478K 1m48s
    50K .......... .......... .......... .......... ..........  0%  477K 1m48s
   100K .......... .......... .......... .......... ..........  0% 30

Archive:  dataset/body_models/smplify.zip
   creating: dataset/body_models/smplify/smplify_public/
   creating: dataset/body_models/smplify/smplify_public/code/
  inflating: dataset/body_models/smplify/smplify_public/code/fit_3d.py  
   creating: dataset/body_models/smplify/smplify_public/code/lib/
   creating: dataset/body_models/smplify/smplify_public/code/models/
  inflating: dataset/body_models/smplify/smplify_public/code/render_model.py  
  inflating: dataset/body_models/smplify/smplify_public/code/show_humaneva.py  
  inflating: dataset/body_models/smplify/smplify_public/code/visualize_mesh_sequence.m  
  inflating: dataset/body_models/smplify/smplify_public/code/visualize_mesh_sequence.py  
  inflating: dataset/body_models/smplify/smplify_public/code/models/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl  
  inflating: dataset/body_models/smplify/smplify_public/code/models/gmm_08.pkl  
  inflating: dataset/body_models/smplify/smplify_public/code/models/regressors_locked_normalized_fe

...

apex is not installed
apex is not installed
apex is not installed
load checkpoint from local path: /kaggle/working/WHAM/checkpoints/vitpose-h-multi-coco.pth
Load backbone weight: /kaggle/working/WHAM/checkpoints/hmr2a.ckpt


Found expected WHAM PKL:
/kaggle/working/WHAM/output/demo/SUB17_E4_wham_rgb_150fps/wham_output.pkl

Saved experiment-specific PKL:
/kaggle/working/wham_pkls/SUB17/E4/SUB17_E4_wham_output.pkl

##########################################################################################
Processing SUB17 E5
Video: /kaggle/input/datasets/sophie160/sub17-experiments/SUB17/E5/SUB17_E5_wham_rgb_150fps.mp4
##########################################################################################

Running command:
/kaggle/working/miniconda/envs/wham/bin/python demo.py --video /kaggle/input/datasets/sophie160/sub17-experiments/SUB17/E5/SUB17_E5_wham_rgb_150fps.mp4 --save_pkl --estimate_local_only


/kaggle/working/miniconda/envs/wham/lib/python3.9/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
2026-06-23 17:02:59.319 | INFO     | __main__:<module>:27 - DPVO is not properly installed. Only estimate in local coordinates !
2026-06-23 17:02:59.332 | INFO     | __main__:<module>:209 - GPU name -> Tesla T4
2026-06-23 17:02:59.332 | INFO     | __main__:<module>:210 - GPU feat -> _CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=14911MB, multi_processor_count=40)
2026-06-23 17:03:05.560 | INFO     | lib.models:build_network:36 - => loaded checkpoint 'checkpoints/wham_vit_bedlam_w_3dpw.pth.tar' 
2026-06-23 17:07:39.747 | INFO     | __main__:run:84 - Complete Data preprocessing!
2026-06-23 17:07:39.754 | INFO     | __main__:run:89 - Save processed data at output/demo/SUB1

apex is not installed
apex is not installed
apex is not installed
load checkpoint from local path: /kaggle/working/WHAM/checkpoints/vitpose-h-multi-coco.pth
Load backbone weight: /kaggle/working/WHAM/checkpoints/hmr2a.ckpt


Found expected WHAM PKL:
/kaggle/working/WHAM/output/demo/SUB17_E5_wham_rgb_150fps/wham_output.pkl

Saved experiment-specific PKL:
/kaggle/working/wham_pkls/SUB17/E5/SUB17_E5_wham_output.pkl

Saved summary CSV:
/kaggle/working/wham_pkls/SUB17/SUB17_wham_pkl_summary.csv

Summary:


,subject,experiment,experiment_number,video_path,wham_original_pkl,metrics_pkl
0,SUB17,E0,0,/kaggle/input/datasets/sophie160/sub17-experim...,/kaggle/working/WHAM/output/demo/SUB17_E0_wham...,/kaggle/working/wham_pkls/SUB17/E0/SUB17_E0_wh...
1,SUB17,E1,1,/kaggle/input/datasets/sophie160/sub17-experim...,/kaggle/working/WHAM/output/demo/SUB17_E1_wham...,/kaggle/working/wham_pkls/SUB17/E1/SUB17_E1_wh...
2,SUB17,E2,2,/kaggle/input/datasets/sophie160/sub17-experim...,/kaggle/working/WHAM/output/demo/SUB17_E2_wham...,/kaggle/working/wham_pkls/SUB17/E2/SUB17_E2_wh...
3,SUB17,E3,3,/kaggle/input/datasets/sophie160/sub17-experim...,/kaggle/working/WHAM/output/demo/SUB17_E3_wham...,/kaggle/working/wham_pkls/SUB17/E3/SUB17_E3_wh...
4,SUB17,E4,4,/kaggle/input/datasets/sophie160/sub17-experim...,/kaggle/working/WHAM/output/demo/SUB17_E4_wham...,/kaggle/working/wham_pkls/SUB17/E4/SUB17_E4_wh...
5,SUB17,E5,5,/kaggle/input/datasets/sophie160/sub17-experim...,/kaggle/working/WHAM/output/demo/SUB17_E5_wham...,/kaggle/working/wham_pkls/SUB17/E5/SUB17_E5_wh...



Created downloadable ZIP:
/kaggle/working/SUB17_wham_pkls.zip

Final PKL files for metrics:
/kaggle/working/wham_pkls/SUB17/E0/SUB17_E0_wham_output.pkl
/kaggle/working/wham_pkls/SUB17/E1/SUB17_E1_wham_output.pkl
/kaggle/working/wham_pkls/SUB17/E2/SUB17_E2_wham_output.pkl
/kaggle/working/wham_pkls/SUB17/E3/SUB17_E3_wham_output.pkl
/kaggle/working/wham_pkls/SUB17/E4/SUB17_E4_wham_output.pkl
/kaggle/working/wham_pkls/SUB17/E5/SUB17_E5_wham_output.pkl

Done.
Use this ZIP for download: /kaggle/working/SUB17_wham_pkls.zip
